# Mosaic — Demo Notebook

## From social structure to accent patterns

This notebook provides a reproducible walkthrough of one complete Mosaic simulation run using the Python API directly — no web interface required.

**Research question:** How does the structure of a social network determine whether speakers' accents converge to a single variety or remain diverse?

**Scope disclaimer:** Mosaic works with synthetic six-dimensional accent vectors. It does not model real speech, real individuals, or real communities. Its purpose is to make social-network assumptions about language change inspectable and reproducible.

---

Run from the repository root with the project virtual environment active:
```bash
pip install -r requirements.txt
jupyter notebook notebooks/demo.ipynb
```

## Model assumptions

Mosaic implements a bounded-confidence accommodation model on a social network.

| Component | Description |
|---|---|
| **Accent vector** | Each agent holds a 6-D vector `d ∈ [0,1]^6` of synthetic phonetic features |
| **Update rule** | At each step one pair is selected; if `‖dᵢ−dⱼ‖ < θ` they partially accommodate |
| **Prestige** | Accommodation toward agent j is scaled by `γ × centrality(j)` |
| **Drift** | Gaussian noise σ is added after each accommodation to prevent lock-in |
| **Convergence** | Run stops early when Shannon diversity over k-means clusters stabilises |

In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/ or repo root

from pprint import pprint
from simulation.config import SimConfig

# Default configuration — Watts-Strogatz small-world, N=200, T=10000
config = SimConfig()
pprint(config.to_dict())

In [ ]:
from simulation.network import make_network
from simulation.config import SimConfig

if 'config' not in locals():
    config = SimConfig()
    
G = make_network(config)
print(f"Topology  : {config.topology}")
print(f"Nodes     : {G.number_of_nodes()}")
print(f"Edges     : {G.number_of_edges()}")
print(f"Avg degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")

communities = set(data['community_id'] for _, data in G.nodes(data=True))
print(f"Communities: {sorted(communities)}")

In [ ]:
from simulation.runner import run_single
from simulation.config import SimConfig

# Use a small config for a fast demo run
demo_config = SimConfig(N=100, T=2000, seed=42)

print("Running simulation...")
metrics = run_single(demo_config)
print()
print(f"Run ID              : {metrics['run_id']}")
print(f"Converged           : {metrics['converged']}")
print(f"Convergence time    : {metrics['convergence_time']} steps")
print(f"Final diversity     : {metrics['final_diversity']:.4f}")
print(f"Final pairwise dist : {metrics['final_pairwise_distance']:.4f}")

In [ ]:
import json
import pathlib
import matplotlib.pyplot as plt

run_dir = pathlib.Path('runs') / metrics['run_id']
with open(str(run_dir / 'timeline.json'), 'r', encoding='utf-8') as f:
    timeline = json.load(f)

timesteps  = [p['timestep']          for p in timeline]
diversity  = [p['diversity']          for p in timeline]
pairwise   = [p['pairwise_distance']  for p in timeline]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(timesteps, diversity, color='#0077ff', linewidth=1.5)
ax1.set_xlabel('Timestep')
ax1.set_ylabel('Shannon diversity (H)')
ax1.set_title('Accent diversity over time')
ax1.grid(True, alpha=0.3)

ax2.plot(timesteps, pairwise, color='#000000', linewidth=1.5)
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Mean pairwise distance (D)')
ax2.set_title('Pairwise accent distance over time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Timeline records: {len(timeline)}")

In [ ]:
import pandas as pd

df = pd.read_csv(str(run_dir / 'agent_states.csv'))
print(f"agent_states.csv shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Logged timesteps: {sorted(df['timestep'].unique())}")
print()

# Final timestep accent distribution
final = df[df['timestep'] == df['timestep'].max()]
print(f"Agents at final timestep: {len(final)}")
print()
print("Community distribution:")
print(final['community_id'].value_counts().to_string())
print()
accent_cols = [f'd{i}' for i in range(6)]
print("Final accent vector statistics:")
print(final[accent_cols].describe().round(4).to_string())

## Reproducibility

Every Mosaic run is fully reproducible from its configuration:

- **Seed:** Setting the same `seed` value produces an identical sequence of random interactions and network structure.
- **Config fingerprint:** Each run writes a `config_fingerprint` (SHA-256 of the canonical config JSON) to `runs/{run_id}/config.json`. Compare fingerprints to confirm two runs used identical parameters.
- **Re-running:** Pass the same `SimConfig` to `run_single()` to reproduce the result exactly.

```python
from simulation.config import SimConfig
from simulation.runner import run_single

config = SimConfig(N=100, T=2000, seed=42)   # same seed → same run
metrics = run_single(config)
```

The `runs/` directory is gitignored — run outputs are not committed to the repository. To share a run, use the export endpoint (`GET /runs/{run_id}/export?format=json`) or copy the `runs/{run_id}/` directory manually.

In [ ]:
# ── Optional: load offline ML benchmark results ─────────────────────────────
# This cell reads pre-computed results from results/ml_results.json.
# It does NOT retrain any model. Skip if the file does not exist.
# To generate: python analysis/train_classifier.py

import pathlib, json
ml_path = pathlib.Path('results/ml_results.json')

if ml_path.exists():
    with open(str(ml_path), 'r', encoding='utf-8') as f:
        ml = json.load(f)
    print("ML benchmark results (pre-computed, not retrained here)")
    print(f"  MLP accuracy : {ml['mlp']['accuracy']:.4f}")
    print(f"  MLP macro-F1 : {ml['mlp']['macro_f1']:.4f}")
    print(f"  GCN accuracy : {ml['gcn']['accuracy']:.4f}")
    print(f"  GCN macro-F1 : {ml['gcn']['macro_f1']:.4f}")
    print(f"  Random chance: {ml['random_chance']:.4f}")
    print(f"  k-means silhouette: {ml['clustering']['kmeans_silhouette']:.4f}")
else:
    print(f"results/ml_results.json not found. Run: python analysis/train_classifier.py")

## Summary

This notebook demonstrated:

1. Creating a `SimConfig` and inspecting default parameters
2. Building the social network graph with `make_network`
3. Running a full simulation with `run_single`
4. Plotting diversity and pairwise distance trajectories from `timeline.json`
5. Loading the `agent_states.csv` snapshot file and inspecting the final accent distribution
6. Reading pre-computed ML benchmark results without retraining

---

### Further exploration

- **Frontend:** Start both servers (`uvicorn api.main:app` + `npm run dev`) and open `http://localhost:5173`
- **API docs:** Open `http://localhost:8000/docs` for the interactive FastAPI Swagger UI
- **Experiments:** See `experiments/` for the four offline experiment scripts
- **ML analysis:** See `analysis/train_classifier.py` to regenerate `results/ml_results.json`

Live site: `http://13.205.223.250` | GitHub Pages showcase: `https://adityawagh19.github.io/Mosaic/`